# AI Coach — POC: Import Garmin FIT Files via USB

Connect your Garmin Forerunner 265 via USB-C. It will appear as a drive in Windows Explorer.  
Copy your `.fit` activity files from `GARMIN/Activity/` on the device into `data/activities/` in this repo, then run the cells below.

In [2]:
from pathlib import Path
import fitparse
import pandas as pd

# --- Config ---
ACTIVITIES_DIR = Path("../data/activities")  # adjust if needed
ACTIVITIES_DIR.mkdir(parents=True, exist_ok=True)

fit_files = sorted(ACTIVITIES_DIR.glob("*.fit"))
print(f"Found {len(fit_files)} .fit file(s):")
for f in fit_files:
    print(f"  {f.name}")

ModuleNotFoundError: No module named 'pandas'

In [ ]:
def parse_fit_file(fit_path: Path) -> pd.DataFrame:
    """Parse a single .fit file and return a DataFrame of 'record' messages."""
    fit = fitparse.FitFile(str(fit_path))
    rows = []
    for msg in fit.get_messages("record"):
        row = {data.name: data.value for data in msg}
        rows.append(row)
    return pd.DataFrame(rows)


# Parse the first available file as a quick test
if fit_files:
    df = parse_fit_file(fit_files[0])
    print(f"Parsed: {fit_files[0].name}  →  {len(df)} records, {len(df.columns)} fields")
    print("\nColumns:", list(df.columns))
    df.head()
else:
    print("No .fit files found. Copy files from your Garmin device to:", ACTIVITIES_DIR.resolve())

In [ ]:
# Parse ALL .fit files into a single combined DataFrame
all_dfs = []
for f in fit_files:
    df_tmp = parse_fit_file(f)
    df_tmp.insert(0, "source_file", f.name)
    all_dfs.append(df_tmp)

if all_dfs:
    df_all = pd.concat(all_dfs, ignore_index=True)
    print(f"Total records across all activities: {len(df_all)}")
    df_all.head()
else:
    print("No .fit files to process.")